# Ориентация текстового фрагмента: 0 или 180 градусов

Для каждого из 20 000 изображений из тестовой выборки необходимо было вычислить вероятность переворота. Основная метрика была Brier score, на которой сильно отражается неверное распознование ориентации текста, поэтому важны именно вероятности, а не только верные классы. Весь путь от подготовки обучающих данных до submission.csv показан ниже. Обычный повторный запуск берёт приложенную дообученную модель и даёт одинаковый файл.

## Почему эта модель и почему её пришлось дообучать


Я использую легковесную модель PP-LCNet_x0_25_textline_ori (https://huggingface.co/PaddlePaddle/PP-LCNet_x0_25_textline_ori_onnx). Выбор пал на неё, так как она изначально предобучена именно на текстовых данных, в отличие от базовых ImageNet-сетей и оптимизирована для быстрого CPU-инференса, что критично при обработке миллионов картинок в день.

Исходно модель уже умеет классифицировать положения строки, но она рассчитана на 4 класса (0, 90, 180, 270 градусов), а не на строгое бинарное распознавание 0/180. Поэтому я заморозил её сверточную часть и в PyTorch переобучил лишь последний линейный слой. Получившийся единый ONNX-файл занимает всего 1.02 МБ. Таким образом, на этапе предсказания не нужны тяжелые OCR-детекторы, полное распознавание текста или внешние API.

На отдельной validation выборке исходная модель получила Brier 0,09175 на смеси и 0,11715 на реальных строках. Это показало заметный доменный сдвиг и раскалибровку уверенности, за которую метрика Brier Score сильно штрафует. Это и стало причиной обучения классификатора. Веса всей модели с нуля я не обучал: признаки уже содержат полезную информацию о шрифтах, а обновление лишь 1 281 значения последнего слоя (при примерно 242 746 значениях весов всей ONNX-модели) требует намного меньше времени, параметров и защищает от переобучения
## Почему 50% реальных и 50% синтетических данных

В train 100 000 изображений, поровну из двух источников: 50 000 реальных строк из HierText (https://github.com/google-research-datasets/hiertext) и 50 000 синтетических строк. Это 25 000 разных исходных кропов каждого источник, каждый присутствует дважды: исходный вариант с меткой 0 и точный поворот на 180° с меткой 1. Поэтому 100 000 примеров — не 100 000 независимых фотографий. Реальные изображения дают шум камеры, сложный фон и настоящую типографику. Синтетика добавляет кириллицу, латиницу, цены, размеры, артикулы и контролируемые искажения. Только реальные данные HierText плохо покрывают такую лексику. Равная пропорция защищает от перекоса в один домен и оставляет оба класса сбалансированными. Это рабочая доля для данного задания, а не доказанный универсальный подход.

Исходные фотографии разделены на train, validation, calibration, selection и holdout до создания поворотных пар. Среди реальных фотографий пересечений между частями нет. Синтетические тексты, шрифты и seed проверки отделены от обучающих. Первоначальные 20 000 изображений нигде не используются для обучения, выбора эпохи или калибровки. Недостающие файлы HierText скачиваются в data/hiertext/, шрифты DejaVu и их лицензия находятся в assets/fonts/.

Ниже показаны проверка данных, код обучения головы, калибровка и построение файла для отправки. В кодовых ячейках комментарии поясняют решения по мере выполнения.

## Среда и фиксированные параметры

SPEC читает сохранённые сведения о модели из model_spec.json. MODEL указывает на готовую ONNX-модель, SELECTED — на выбранные настройки калибровки.

RUN_TRAINING = False/True выбор обучать модель или нет.

CONFIG хранит параметры: seed 100, размер входа 80×160, размеры пакетов, число эпох и размер выборки. device='cpu' задаёт обучение на CPU.

In [27]:
from pathlib import Path
import hashlib
import json
import random
import time

import numpy as np
import pandas as pd
from PIL import Image
import torch

from src.pplcnet_head import preprocess, session, sha256
from src.calibration import calibrate, safe_logit

SPEC = json.loads(Path('model_spec.json').read_text(encoding='utf-8'))
MODEL = Path('results/pplcnet_x0_25/fine_tuned.onnx')
SELECTED = Path('results/pplcnet_x0_25/selected.json')
RUN_TRAINING = False  # Включайте только для полного повторения обучения с исходных архивов.
CONFIG = dict(seed=100, image_size=(80, 160), feature_batch=64,
              train_batch=1024, epochs=100, lr=1e-3, weight_decay=1e-4,
              num_workers=0, train_size=100_000, validation_size=4_000,
              model_name='PP-LCNet_x0_25_textline_ori', device='cpu')

random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
if not RUN_TRAINING:
    assert sha256(MODEL) == SPEC['fine_tuned_model_sha256']
print(f'Модель: {CONFIG["model_name"]}, размер: {round(MODEL.stat().st_size / 1e6, 2)} МБ')


Модель: PP-LCNet_x0_25_textline_ori, размер: 1.02 МБ


## Проверяем данные

Проверяем, что тестовые изображения соответствуют sample_submission.csv, открываются без ошибок и не дублируются, показывает их размеры, а также проверяет состав и баланс обучающей выборки.

In [28]:
sample = pd.read_csv('sample_submission.csv', dtype={'image_id': str})
paths = {p.stem: p for p in Path('test/images').glob('*.png')}
assert list(sample.columns) == ['image_id', 'p_180']
assert len(sample) == len(paths) == 20_000
assert sample.image_id.is_unique and set(sample.image_id) == set(paths)

# Проверяем открытие каждого PNG, цветовой режим, размеры и дубликаты по содержимому.
eda_rows, fingerprints = [], set()
for path in paths.values():
    with Image.open(path) as image:
        image.load()
        eda_rows.append((image.width, image.height, image.mode))
    fingerprint = sha256(path)
    assert fingerprint not in fingerprints, f'Повторяющийся файл: {path}'
    fingerprints.add(fingerprint)
eda = pd.DataFrame(eda_rows, columns=['width', 'height', 'mode'])
eda['aspect_ratio'] = eda.width / eda.height
assert eda['mode'].value_counts().to_dict() == {'RGB': 20_000}
display(eda[['width', 'height', 'aspect_ratio']].describe().round(2))

train = pd.read_csv('results/mixed/manifests/train_manifest.csv')
assert len(train) == 100_000 and train.base_id.nunique() == 50_000
assert train.source.value_counts().to_dict() == {'real': 50_000, 'synthetic': 50_000}
assert train.label.value_counts().to_dict() == {0: 50_000, 1: 50_000}
assert train.groupby('base_id').label.agg(set).map(lambda labels: labels == {0, 1}).all()
print(f'Обучающая выборка содержит {len(train)} изображений, {train.base_id.nunique()} исходных кропов')
print(f'Тестовая выборка содержит {len(paths)} изображений, порядок задаёт sample_submission.csv')

,width,height,aspect_ratio
count,20000.00,20000.00,20000.00
mean,357.60,64.40,6.22
std,318.93,59.34,4.27
min,25.00,10.00,0.72
25%,129.00,27.00,3.50
50%,238.00,42.00,4.89
75%,479.25,76.00,7.52
max,1599.00,492.00,49.67


Обучающая выборка содержит 100000 изображений, 50000 исходных кропов
Тестовая выборка содержит 20000 изображений, порядок задаёт sample_submission.csv


## Подготовка и обучение

Если нужно проверить обучение с самого начала, переключатель RUN_TRAINING скачает закреплённые версии аннотаций и фотографий HierText, проверит их SHA256, создаст реальные кропы и синтетические строки, затем извлечёт 1280-мерные признаки замороженной PP-LCNet. На этом шаге модель видит только train. Размер входа 80×160 и нормализация взяты из официального inference.yml.

Линейная голова стартует с официальных весов классификатора, а не со случайной инициализации. Обучаю её BCEWithLogitsLoss и AdamW (lr=0.001, weight_decay=0.0001, batch 1024). После каждой эпохи считаю Brier отдельно на реальных и синтетических данных validation. Для выбора checkpoint использую 0.7 × Brier_real + 0.3 × Brier_synthetic: итоговый поток ближе к реальным фотографиям, поэтому их ошибки важнее. Остановка — после четырёх эпох без улучшения. Эта валидация не участвует в градиентном шаге.

Здесь приведён сам цикл обучения. Извлечение признаков и подготовка данных находятся в небольших файлах src/ рядом с блокнотом; история каждой эпохи сохранена в JSON.

Если включить обучение, в конце блокнота обновятся метрики и контрольные суммы в model_spec.json.

In [29]:
from src.pplcnet_head import extract_split, official_head, score, RUN_DIR

def fit_head() -> None:
    torch.manual_seed(CONFIG['seed'])
    torch.set_num_threads(4)
    # Извлекаю признаки один раз: основную часть модели не обучаю.
    x_train, y_train, _ = extract_split("train")
    x_val, y_val, val_frame = extract_split("val")
    val_real = val_frame.source.eq("real").to_numpy()
    w0, b0 = official_head()
    x = torch.from_numpy(np.asarray(x_train).copy())
    y = torch.from_numpy(y_train).reshape(-1, 1)
    xv = torch.from_numpy(np.asarray(x_val).copy())
    model = torch.nn.Linear(1280, 1)
    # Беру начальные веса головы из официальной модели.
    with torch.no_grad():
        model.weight.copy_(torch.from_numpy(w0[None, :]))
        model.bias.fill_(b0)
    print("Исходная модель:", score(y_val, x_val @ w0 + b0), flush=True)
    print("Исходная модель, реальные:", score(y_val[val_real], (x_val @ w0 + b0)[val_real]), flush=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best, best_state, wait = float("inf"), None, 0
    history = []
    for epoch in range(1, 101):
        model.train()
        order = torch.randperm(len(x), generator=torch.Generator().manual_seed(CONFIG['seed'] + epoch))
        losses = []
        for indices in order.split(1024):
            optimizer.zero_grad(set_to_none=True)
            logits = model(x[indices])
            loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y[indices])
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach()))
        model.eval()
        with torch.no_grad():
            val_logits = model(xv).flatten().numpy()
        overall = score(y_val, val_logits)
        real = score(y_val[val_real], val_logits[val_real])
        synthetic = score(y_val[~val_real], val_logits[~val_real])
        # Для проверки доменного сдвига реальные кропы получают больший вес.
        criterion = .7 * real["brier"] + .3 * synthetic["brier"]
        record = {"epoch": epoch, "train_bce": float(np.mean(losses)),
                  "criterion": criterion, "val": overall, "real": real, "synthetic": synthetic}
        history.append(record)
        print(json.dumps(record, ensure_ascii=False), flush=True)
        if criterion < best - 1e-5:
            # Запоминаю эпоху с лучшим Brier на validation.
            best, best_state, wait = criterion, (model.weight.detach().clone(), model.bias.detach().clone()), 0
        else:
            wait += 1
            if wait >= 4:
                break
    assert best_state is not None
    out = RUN_DIR / "head.npz"
    np.savez(out, weight=best_state[0].numpy().reshape(-1), bias=best_state[1].numpy())
    (RUN_DIR / "training_history.json").write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Сохранено:", out, "SHA256:", sha256(out), flush=True)

In [30]:
if RUN_TRAINING:
    from src.config import MIXED_CONFIG
    from src.mixed_data import mixed_manifests
    from src.pplcnet_head import export_head, choose_calibration, evaluate_exported_holdout

    # Сначала разделяем исходные фотографии, затем создаём пары 0°/180°.
    mixed_manifests(MIXED_CONFIG)
    # Извлечение признаков может занять несколько минут; сеть не меняется.
    fit_head()
    choose_calibration()
    export_head()
    # Проверяю уже экспортированную ONNX-модель тем же способом, что и тест.
    evaluate_exported_holdout()
    print('SHA256 модели после нового обучения:', sha256(MODEL))

    history = pd.DataFrame(json.loads(Path('results/pplcnet_x0_25/training_history.json').read_text(encoding='utf-8')))
    display(history[['epoch', 'train_bce', 'criterion']].round(5))
    best_epoch = int(history.loc[history.criterion.idxmin(), 'epoch'])
    print('Лучшая эпоха:', best_epoch)

,epoch,train_bce,criterion
0,1,0.28193,0.07185
1,2,0.20522,0.05985
2,3,0.18876,0.05616
3,4,0.18113,0.05367
4,5,0.17570,0.05198
...,...,...,...
77,78,0.14651,0.04359
78,79,0.14622,0.04363
79,80,0.14623,0.04367
80,81,0.14622,0.04359


Лучшая эпоха: 82


## Вероятности и независимая проверка

Голова выдаёт два логита, их softmax даёт исходную вероятность поворота. Для Brier это ещё не обязательно хорошая вероятность: модель может быть уверенной слишком сильно. На отдельном calibration-наборе подбираю температуру, а на selection-наборе проверяю, помогает ли усреднение p(x) и 1 − p(R180(x)). Holdout не участвовал ни в обучении, ни в выборе настроек.

In [31]:
selected = json.loads(SELECTED.read_text(encoding='utf-8'))
print('Калибровка:', selected['method'], '| поворотное усреднение:', selected['tta'])
if RUN_TRAINING:
    holdout = pd.read_csv('results/pplcnet_x0_25/holdout_exported_metrics.csv')
    display(holdout.round(5))

Калибровка: temperature | поворотное усреднение: True


,source,images,brier,accuracy,auc
0,all,2000,0.04290,0.940,0.98917
1,real,1000,0.04816,0.934,0.98671
2,synthetic,1000,0.03763,0.946,0.99156


## Предсказания и файл для отправки

Используем только классификатор строк. Читаем test в порядке sample_submission.csv, применяем тот же препроцессинг, затем калибровку. Точность записи зафиксирована до восьми знаков, в конце проверяем состав и SHA256 файла.

In [32]:
predictor = session(MODEL)
probabilities = []
start = time.perf_counter()
for offset in range(0, len(sample), 32):
    images = []
    for image_id in sample.image_id.iloc[offset:offset + 32]:
        with Image.open(paths[image_id]) as image:
            images.append(image.convert('RGB'))
    prepared = np.stack([preprocess(image) for image in images])
    # Вероятность класса 1 соответствует повороту на 180°.
    raw = predictor.run(None, {'x': prepared})[0][:, 1]
    p = calibrate(safe_logit(raw), selected['calibration'])
    if selected['tta']:
        # Это тот же поворот после линейного resize, но без повторной обработки PNG.
        rotated = predictor.run(None, {'x': prepared[:, :, ::-1, ::-1].copy()})[0][:, 1]
        p = .5 * (p + 1 - calibrate(safe_logit(rotated), selected['calibration']))
    probabilities.extend(p.tolist())

# Сохраняю строки в порядке sample_submission.csv.
submission = pd.DataFrame({'image_id': sample.image_id, 'p_180': probabilities})
assert len(submission) == 20_000 and submission.image_id.is_unique
assert submission.p_180.notna().all() and submission.p_180.between(0, 1).all()
assert submission.image_id.tolist() == sample.image_id.tolist()
submission.to_csv('submission.csv', index=False, float_format='%.8f', lineterminator='\n')
if RUN_TRAINING:
    metrics = holdout.set_index('source')
    SPEC.update(seed=CONFIG['seed'], fine_tuned_model_sha256=sha256(MODEL),
                submission_sha256=sha256(Path('submission.csv')), best_epoch=best_epoch,
                holdout_brier=float(metrics.loc['all', 'brier']),
                holdout_accuracy=float(metrics.loc['all', 'accuracy']),
                holdout_real_brier=float(metrics.loc['real', 'brier']))
    Path('model_spec.json').write_text(json.dumps(SPEC, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
else:
    assert sha256(Path('submission.csv')) == SPEC['submission_sha256']
print('Готово:', len(submission), 'строк;', round(len(submission)/(time.perf_counter()-start)), 'изображений/с')
display(submission.head())

Готово: 20000 строк; 449 изображений/с


,image_id,p_180
0,test_00000,0.024072
1,test_00001,0.999994
2,test_00002,0.999988
3,test_00003,0.131344
4,test_00004,0.936709


## Вывод

На тестовой выборке модель получила 1 − Brier Score = 0,94383094 (Brier Score 0,05616906). При этом итоговая ONNX-модель занимает около 1,02 МБ и обрабатывает 449 изображений/с на CPU.

Результат хороший, но модель не идеальна: тестовый Brier выше, чем на моей отложенной выборке (0,04290). Для улучшения я бы сначала разобрал ошибки на реальных кропах и проверил калибровку вероятностей, а затем попробовал добавить более разнообразные реальные изображения в обучение, разный % реальных и синтетических данных.